# History of Clayton Kershaw
## Use this notebook to practice the following
    1. Use AI to generate a biography on Clayton Kershaw
    2. Use Tools to store all questions to the database
    3. Use Gradio to enable us to communicate with the AI about CK
#

In [45]:
from openai import OpenAI
from dotenv import load_dotenv
import json

In [46]:
# load env and create client
load_dotenv()

oaClient = OpenAI()

In [47]:
biography_prompt = "Generate a two paragraphs about Clayton Kershaw. ^ Include information about his debut, accomplishments, best and worst season, number of homeruns, etc."

In [48]:
messages = [{"role":"user", "content":biography_prompt}]

In [49]:
response = oaClient.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages
)

In [50]:
system_prompt = f"You are teaching users about the history of Clayton Kershaw;  you are only allowed to answer regarding the information in the prompt provided.  If you do not know the answer to anything, you must inform the user that you are unaware of the answer.  In addition, if you do not know the answer, send the question to the output_question tool with is_known set to False.  For all questions you know the answers for, send them to the output tool with is_known set to True.  Prompt: {txt}"

In [77]:
def output_question(message, is_known):
    print(f"logging question: {is_known}: {message}")

output_question_json = {
    "name": "output_question",
    "description": "Output every question using this tool.  Send the question in the message parameter, and set is_known to True or False depending on if you know the answer",
    "paramaters":{
    "message": {
        "type":"string",
        "description":"The actual question being asked",
    },
    "is_known":{
          "type":"bool",
         "description":"False when you do not know the answer, True when you do know the answer" ,
         },
        "required": ["message", "is_known"],
        "additionalProperties": False
    },
    }

tools = [{"type":"function", "function":output_question_json}]



In [ ]:
def handle_tool_calls(calls):
    for call in calls:
        fn_name= call.function.name
        args = call.function.arguments
        print(args)
        #return {"role:":"tools", "content":json.dumps({"recorded":"ok"}), "tool_call_id":call.id}
        return [{"role":"tool", "content":json.dumps({"recorded": "ok"}), "tool_call_id": call.id}]

        #fn = globals().get(fn_name)
        #result = fn(**args)
        #print(result)


In [96]:
def gradioCallback(message, history):
    done = False
    # put together another prompt 
    messages = [{"role":"system", "content": system_prompt}, {"role":"user","content": message}]
    while not done:
        response = oaClient.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools,
        )

        finish_reason = response.choices[0].finish_reason
        if finish_reason =="tool_calls":
            print("CALLING TOOLS")
            result = handle_tool_calls(response.choices[0].message.tool_calls)
            messages.append(response.choices[0].message)
            message.extend(result)
            #messages.extend([{"role":"tool", "content":json.dumps({"recorded": "ok"}), "tool_call_id": response.choices[0].message.tool_calls[0].id}])
        else:
            print("NO TOOLS TO CALL")
            done = True

    print("done")

   # tool_calls = response.choices[0].message.tool_calls
   # handle_tool_calls(tool_calls)
    return response.choices[0].message.content

In [97]:
import gradio as gr

gr.ChatInterface(gradioCallback, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7896
* To create a public link, set `share=True` in `launch()`.
